In [1]:
import os
import torch
import cv2
import numpy as np
from PIL import Image
from tqdm import tqdm
from collections import defaultdict
from segment_anything import sam_model_registry, SamPredictor

# -------------------------
# CONFIGURATION
# -------------------------
image_dir = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_images"
point_mask_dir = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_point_masks_all_regions_1"
output_dir = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/SAM_predictions_boundary"
os.makedirs(output_dir, exist_ok=True)

sam_checkpoint = "/home/iiitdmk-param/AMB/Proposed_3/SAM/sam_vit_h_4b8939.pth"
model_type = "vit_h"
device = "cuda" if torch.cuda.is_available() else "cpu"

IGNORE_RGB = (100, 100, 100)  # unlabeled background

# -------------------------
# HELPER
# -------------------------
def get_boundary_from_mask(mask):
    mask = mask.astype(np.uint8) * 255
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boundary = np.zeros_like(mask)
    cv2.drawContours(boundary, contours, -1, 255, thickness=1)
    return boundary

# -------------------------
# LOAD SAM
# -------------------------
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device)
predictor = SamPredictor(sam)

# -------------------------
# MAIN LOOP
# -------------------------
for fname in tqdm(os.listdir(image_dir)):
    if not fname.endswith((".png", ".jpg", ".jpeg")):
        continue

    img_path = os.path.join(image_dir, fname)
    mask_path = os.path.join(point_mask_dir, fname)
    output_path = os.path.join(output_dir, fname)

    image = cv2.imread(img_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    point_mask = np.array(Image.open(mask_path))
    h, w = point_mask.shape[:2]
    predictor.set_image(image_rgb)

    # Step 1: Group points by class color
    color_to_points = defaultdict(list)
    for y in range(h):
        for x in range(w):
            px = tuple(point_mask[y, x])
            if px == IGNORE_RGB:
                continue
            color_to_points[px].append([x, y])  # x,y format for SAM

    final_boundary = np.zeros((h, w), dtype=np.uint8)

    # Step 2: For each color group, merge their SAM masks
    for color, point_list in color_to_points.items():
        merged_mask = np.zeros((h, w), dtype=np.uint8)

        for point in point_list:
            masks, _, _ = predictor.predict(
                point_coords=np.array([point]),
                point_labels=np.array([1]),
                multimask_output=False
            )
            mask = masks[0].astype(np.uint8)
            merged_mask = np.logical_or(merged_mask, mask)

        # Step 3: Extract boundary of merged mask
        boundary = get_boundary_from_mask(merged_mask)
        final_boundary = np.maximum(final_boundary, boundary)

    # Step 4: Save boundary map
    cv2.imwrite(output_path, final_boundary)


/home/iiitdmk-param/AMB/Proposed_3/SAM/segment-anything-main/segment_anything/build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)

00%

In [2]:
import os
import cv2
import numpy as np

# --- CONFIGURATION ---

# 1. DEFINE YOUR FOLDER PATHS
BOUNDARY_FOLDER = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/SAM_predictions_boundary"
MASK_FOLDER = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/train_point_masks_all_regions_1"
OUTPUT_FOLDER = "/home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/merged_masks/"

# 2. DEFINE THE COLOR MAP (BGR format for OpenCV)
COLOR_MAP = {
    'airplane': (240, 202, 166),
    'bare soil': (0, 128, 128),
    'buildings': (128, 0, 0),
    'cars': (0, 0, 255),
    'chaparral': (0, 128, 0),
    'court': (0, 0, 128),
    'dock': (233, 233, 255),
    'field': (164, 160, 160),
    'grass': (128, 128, 0),
    'mobile home': (255, 87, 90),
    'pavement': (0, 255, 255),
    'sand': (0, 192, 255),
    'sea': (255, 0, 0),
    'ship': (92, 0, 255),
    'tanks': (128, 0, 128),
    'trees': (0, 255, 0),
    'water': (255, 255, 0),
}

# Reverse lookup: from BGR to class name
COLOR_NAMES = {v: k for k, v in COLOR_MAP.items()}

# 3. BACKGROUND COLOR (for unlabeled / ignore pixels)
BACKGROUND_COLOR = (100, 100, 100)  # grey background

# --- SCRIPT LOGIC ---

def find_dominant_color(region_mask, mask_image):
    """
    Finds the most frequent class color in a region of mask_image,
    ignoring background.
    """
    pixels = mask_image[region_mask == 255]
    unique_colors, counts = np.unique(pixels.reshape(-1, 3), axis=0, return_counts=True)

    dominant_color = None
    max_count = -1
    for i, color_tuple in enumerate(unique_colors):
        color = tuple(color_tuple)
        if color == BACKGROUND_COLOR:
            continue
        if color in COLOR_NAMES and counts[i] > max_count:
            max_count = counts[i]
            dominant_color = color
    return dominant_color

def process_images():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    boundary_files = os.listdir(BOUNDARY_FOLDER)

    for boundary_filename in boundary_files:
        boundary_path = os.path.join(BOUNDARY_FOLDER, boundary_filename)
        mask_path = os.path.join(MASK_FOLDER, boundary_filename)
        output_path = os.path.join(OUTPUT_FOLDER, boundary_filename)

        if not os.path.exists(mask_path):
            print(f"⚠️ No matching mask for {boundary_filename}, skipping.")
            continue

        print(f"\nProcessing: {boundary_filename}")
        boundary_img_bgr = cv2.imread(boundary_path)
        mask_img = cv2.imread(mask_path)

        if boundary_img_bgr is None or mask_img is None:
            print(f"Error loading {boundary_filename}, skipping.")
            continue

        boundary_img_gray = cv2.cvtColor(boundary_img_bgr, cv2.COLOR_BGR2GRAY)
        contours, _ = cv2.findContours(boundary_img_gray, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        color_to_contours = {}
        for contour in contours:
            contour_mask = np.zeros(boundary_img_gray.shape, dtype=np.uint8)
            cv2.drawContours(contour_mask, [contour], -1, 255, thickness=cv2.FILLED)
            dominant_color = find_dominant_color(contour_mask, mask_img)
            if dominant_color:
                color_name = COLOR_NAMES.get(dominant_color, "Unknown")
                color_to_contours.setdefault(color_name, []).append(contour)

        output_image = np.zeros_like(boundary_img_bgr)
        for color_name, contour_list in color_to_contours.items():
            print(f"Found {len(contour_list)} region(s) for {color_name}")
            if len(contour_list) > 1:
                all_points = np.vstack(contour_list)
                merged_boundary = cv2.convexHull(all_points)
                cv2.drawContours(output_image, [merged_boundary], -1, (255, 255, 255), 1)
                print("  → Merged into single boundary.")
            else:
                cv2.drawContours(output_image, contour_list, -1, (255, 255, 255), 1)

        cv2.imwrite(output_path, output_image)
        print(f"✅ Saved: {output_path}")

if __name__ == '__main__':
    process_images()



Processing: 251.png
Found 1 region(s) for cars
✅ Saved: /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/merged_masks/251.png

Processing: 542.png
Found 2 region(s) for buildings
  → Merged into single boundary.
Found 2 region(s) for trees
  → Merged into single boundary.
✅ Saved: /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/merged_masks/542.png

Processing: 8.png
Found 1 region(s) for field
✅ Saved: /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/merged_masks/8.png

Processing: 233.png
Found 1 region(s) for trees
✅ Saved: /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/merged_masks/233.png

Processing: 611.png
Found 1 region(s) for buildings
Found 1 region(s) for court
✅ Saved: /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/merged_masks/611.png

Processing: 302.png
Found 1 region(s) for water
✅ Saved: /home/iiitdmk-param/A.C/Anisha/4th objective/dlrsd/merged_masks/302.png

Processing: 73.png
Found 3 region(s) for buildings
  → Merged into single boundary.
✅ Saved: /